# Install package

In [ ]:
!pip install pygamma-agreement

In [ ]:
!pip install "pygamma-agreement[CBC]"

# Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Imports

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from pygamma_agreement import Continuum
from pyannote.core import Segment
from pygamma_agreement import CombinedCategoricalDissimilarity

# Find the drive folder with all the files

In [ ]:
os.listdir("/content/drive/MyDrive/Zip files/ALL_JSON_files_MAthesis")

# JSON parser


In [ ]:
def parse_inception_json(filepath, annotator_name):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    feature_structures = data["%FEATURE_STRUCTURES"]

    # Map internal INCEpTION names to readable label names
    # Example: IClaim -> C-IMP, Narrative -> Exordium
    feature_name_to_ui_name = {}

    for fs in feature_structures:
        if fs.get("%TYPE") == "de.tudarmstadt.ukp.clarin.webanno.api.type.FeatureDefinition":
            internal_name = fs.get("name")
            readable_name = fs.get("uiName")

            if internal_name and readable_name:
                feature_name_to_ui_name[internal_name] = readable_name

    # These are text helper fields, not span labels for gamma
    ignore_features = {
        "IClaimtext",
        "MCIMPtext",
        "PIMPtext"
    }

    # These are the actual annotation layers
    annotation_layers = {
        "webanno.custom.Argumentation",
        "webanno.custom.Narration"
    }

    annotations = []

    for fs in feature_structures:
        if fs.get("%TYPE") not in annotation_layers:
            continue

        start = fs.get("begin")
        end = fs.get("end")

        if start is None or end is None:
            continue

        for feature_name, value in fs.items():
            if feature_name.startswith("%") or feature_name.startswith("@"):
                continue

            if feature_name in {"begin", "end"}:
                continue

            if feature_name in ignore_features:
                continue

            if value is True:
                label = feature_name_to_ui_name.get(feature_name, feature_name)

                annotations.append({
                    "annotator": annotator_name,
                    "start": int(start),
                    "end": int(end),
                    "label": label
                })

    return annotations

In [ ]:
annotation_root = "/content/drive/MyDrive/Zip files/ALL_JSON_files_MAthesis/annotation"

# Define label groups for argumentation layer and narration layer

In [ ]:
argumentation_labels = {
    "C-EXP",
    "MC-EXP",
    "C-IMP",
    "MC-IMP",
    "P-EXP",
    "P-IMP"
}

narration_labels = {
    "Confirmatio",
    "Narratio",
    "Exordium",
    "Other",
    "Peroratio",
    "Propositio"
    "NONE"
}

# Helper function
 So we can use it for all gamma tests: overall, argumentation, narration, standard, soft, and stronger label penalty.

In [ ]:
def compute_gamma_test(
    annotations,
    layer_name="overall",
    selected_labels=None,
    alpha=1,
    beta=1,
    soft=False,
    seed=42,
    add_none_for_missing_narration=False
):
    continuum = Continuum()

    selected_annotations = [
        ann for ann in annotations
        if selected_labels is None or ann["label"] in selected_labels
    ]

    # Special case: narration layer where one annotator has no narration annotations.
    # We add a dummy "none" span so gamma can still compare two annotators.
    if add_none_for_missing_narration:
        annotators_present = {ann["annotator"] for ann in selected_annotations}
        all_annotators = {ann["annotator"] for ann in annotations}

        missing_annotators = all_annotators - annotators_present

        for annotator in missing_annotators:
            selected_annotations.append({
                "annotator": annotator,
                "start": 0,
                "end": 1,
                "label": "none"
            })

    for ann in selected_annotations:
        continuum.add(
            ann["annotator"],
            Segment(ann["start"], ann["end"]),
            ann["label"]
        )

    dissimilarity = CombinedCategoricalDissimilarity(
        alpha=alpha,
        beta=beta
    )

    np.random.seed(seed)

    gamma_result = continuum.compute_gamma(
        dissimilarity,
        soft=soft
    )

    return {
        "layer": layer_name,
        "gamma_type": "soft" if soft else "standard",
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma_result.gamma
    }

# TED folders with both annotations

In [ ]:
document_pairs = []

# Go through every folder inside annotation/
for folder_name in os.listdir(annotation_root):

    folder_path = os.path.join(annotation_root, folder_name)

    # Only keep folders
    if not os.path.isdir(folder_path):
        continue

    # Build paths to both annotators
    luiza_file = os.path.join(folder_path, "filip.json")
    sara_file = os.path.join(folder_path, "nabhani.json")

    # Only keep folders where BOTH files exist
    if os.path.exists(luiza_file) and os.path.exists(sara_file):

        document_pairs.append({
            "document": folder_name.replace(".txt", ""),
            "luiza_file": luiza_file,
            "sara_file": sara_file
        })

print("Documents found:", len(document_pairs))


# Compute Gamma for everything

In [ ]:
all_results = []

for pair in document_pairs:
    document_id = pair["document"]

    print("Processing:", document_id)

    luiza_annotations = parse_inception_json(
        pair["luiza_file"],
        "Luiza"
    )

    sara_annotations = parse_inception_json(
        pair["sara_file"],
        "Sara"
    )

    all_annotations = luiza_annotations + sara_annotations

    gamma_tests = [

        # Overall standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=1,
            soft=False),

        # Overall soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=1,
            soft=True),

        # Overall standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=2,
            soft=False),

        # Overall soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=2,
            soft=True),

        # Arguemntation standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=False),

        # Arguemntation soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=True),

        # Arguemntation standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=False),

        # Arguemntation soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=True),

        # Narration standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=1,
            soft=False,
            add_none_for_missing_narration=True),


        # Narration soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=1,
            soft=True,
            add_none_for_missing_narration=True),

        # Narration standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=2,
            soft=False,
            add_none_for_missing_narration=True),

        # Narration soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=2,
            soft=True,
            add_none_for_missing_narration=True),
    ]


    for result in gamma_tests:
        result["document"] = document_id
        result["luiza_annotations"] = len(luiza_annotations)
        result["sara_annotations"] = len(sara_annotations)
        all_results.append(result)

# Save all result in a table

In [ ]:
all_results_df = pd.DataFrame(all_results)

all_results_df = all_results_df[
    [
        "document",
        "layer",
        "gamma_type",
        "alpha",
        "beta",
        "gamma",
        "luiza_annotations",
        "sara_annotations"
    ]
]

all_results_df

In [ ]:
output_path = "/content/drive/MyDrive/ALL_gamma_results_teds.csv"

all_results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

# EXTRA: only argumentation Gamma

In [ ]:
only_arg_results = []

for pair in document_pairs:
    document_id = pair["document"]

    print("Processing:", document_id)

    luiza_annotations = parse_inception_json(
        pair["luiza_file"],
        "Luiza"
    )

    sara_annotations = parse_inception_json(
        pair["sara_file"],
        "Sara"
    )

    all_annotations = luiza_annotations + sara_annotations

    gamma_tests = [

        # Arguemntation standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=False),

        # Arguemntation soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=True),

        # Arguemntation standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=False),

        # Arguemntation soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=True),
    ]


    for result in gamma_tests:
        result["document"] = document_id
        result["luiza_annotations"] = len(luiza_annotations)
        result["sara_annotations"] = len(sara_annotations)
        only_arg_results.append(result)

# Save in a table

In [ ]:
only_arg_results_df = pd.DataFrame(all_results)

only_arg_results_df = only_arg_results_df[
    [
        "document",
        "layer",
        "gamma_type",
        "alpha",
        "beta",
        "gamma",
        "luiza_annotations",
        "sara_annotations"
    ]
]

only_arg_results_df

In [ ]:
output_path = "/content/drive/MyDrive/ONLY_ARG_gamma_results_teds.csv"

only_arg_results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)